# 05 -- Vanilla RNN with Pretrained Embeddings (Step 3, my part)

My teammate's notebook (`03_lstm_attention.ipynb`) covers the LSTM + attention model. This notebook is the plain RNN counterpart, kept consistent with their setup so the final comparison across models is fair:
- Loads the same `cleaned_bfrd.csv` from `/content/drive/MyDrive/BanglaFakeReviewDetector/`
- Uses the same preprocessing functions
- Uses `random_state=42` for an identical train/val/test split
- Uses the same cached pretrained fastText Bangla vectors
- Reports the same metrics: accuracy, precision, recall, F1, macro-F1

In [1]:
!pip install -q gensim torch scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 28.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/BanglaFakeReviewDetector'
os.chdir(PROJECT_DIR)

import pandas as pd
df = pd.read_csv("cleaned_bfrd.csv", encoding="utf-8-sig")
counts = df["label"].value_counts()
AUTHENTIC_LABEL, FAKE_LABEL = counts.idxmax(), counts.idxmin()
print(f"Loaded {len(df)} rows. authentic={AUTHENTIC_LABEL!r}, fake={FAKE_LABEL!r}")

Mounted at /content/drive
Loaded 9032 rows. authentic=np.int64(1), fake=np.int64(0)


## Preprocessing (identical to the other notebooks, for a fair comparison)

In [3]:
import re
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

BANGLA_STOPWORDS = set(stopwords.words('bengali'))
_porter = PorterStemmer()

EMOJI_PATTERN = re.compile(
    "[" "\U0001F300-\U0001FAFF" "\U00002600-\U000027BF"
    "\U0001F1E6-\U0001F1FF" "\U00002190-\U000021FF" "]+", flags=re.UNICODE)

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = EMOJI_PATTERN.sub(" ", text)
    text = re.sub(r"([\u0964!?.\-_=~*])\1{1,}", r"\1", text)
    text = re.sub(r"(#\w+)(\s*\1){1,}", r"\1", text)
    return re.sub(r"\s+", " ", text).strip()

def is_bangla_token(token):
    if not token:
        return False
    bangla_chars = sum(1 for ch in token if "\u0980" <= ch <= "\u09FF")
    return bangla_chars > len(token) / 2

def basic_tokenize(text):
    return re.findall(r"[\u0980-\u09FFa-zA-Z0-9]+", text)

def remove_stopwords(tokens):
    return [t for t in tokens if t not in BANGLA_STOPWORDS]

def stem_latin_tokens(tokens):
    return [_porter.stem(t) if not is_bangla_token(t) else t for t in tokens]

def preprocess_review(raw_text):
    cleaned = clean_text(raw_text)
    tokens = basic_tokenize(cleaned)
    return {"cleaned_text": cleaned, "tokens": stem_latin_tokens(remove_stopwords(tokens))}

print(f"Loaded {len(BANGLA_STOPWORDS)} Bengali stopwords.")

Loaded 398 Bengali stopwords.


## Load pretrained Bangla embeddings (same cached file on Drive as the LSTM notebook)

In [4]:
import urllib.request, os
from gensim.models import KeyedVectors

FASTTEXT_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.bn.300.vec.gz"
FASTTEXT_PATH = "cc.bn.300.vec.gz"  # same file the LSTM notebook downloads/caches -- reused here

if not os.path.exists(FASTTEXT_PATH):
    print("Downloading fastText Bangla vectors (~840MB, one-time -- saved to Drive after)...")
    urllib.request.urlretrieve(FASTTEXT_URL, FASTTEXT_PATH)
    print("Done.")
else:
    print("Found cached vectors on Drive -- skipping download.")

kv = KeyedVectors.load_word2vec_format(FASTTEXT_PATH, binary=False, limit=200_000)
EMBED_DIM = kv.vector_size
print(f"Loaded {len(kv)} pretrained vectors, dim={EMBED_DIM}")

Found cached vectors on Drive -- skipping download.
Loaded 200000 pretrained vectors, dim=300


## Build vocabulary, embedding matrix, and the vanilla RNN model

In [5]:
import random
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score

RANDOM_STATE = 42
random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE); torch.manual_seed(RANDOM_STATE)

MAX_VOCAB_SIZE, MAX_SEQ_LEN = 8000, 40
PAD_TOKEN, UNK_TOKEN = "<pad>", "<unk>"

def build_vocab(tokenized_texts, max_size=MAX_VOCAB_SIZE):
    counts = Counter(tok for tokens in tokenized_texts for tok in tokens)
    vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    for tok, _ in counts.most_common(max_size - 2):
        vocab[tok] = len(vocab)
    return vocab

def encode(tokens, vocab, max_len=MAX_SEQ_LEN):
    ids = [vocab.get(t, vocab[UNK_TOKEN]) for t in tokens[:max_len]]
    return ids + [vocab[PAD_TOKEN]] * (max_len - len(ids))

def build_embedding_matrix(vocab, kv, embed_dim):
    matrix = np.random.normal(scale=0.1, size=(len(vocab), embed_dim)).astype(np.float32)
    matrix[0] = np.zeros(embed_dim, dtype=np.float32)
    found = 0
    for word, idx in vocab.items():
        if word in kv:
            matrix[idx] = kv[word]
            found += 1
    print(f"Found pretrained vectors for {found}/{len(vocab)} words ({100*found/len(vocab):.1f}%).")
    return matrix

class RNNClassifier(nn.Module):
    """Vanilla (Elman) RNN with pretrained embeddings and masked mean-pooling over timesteps.
    This is the plain-RNN counterpart to the teammate's LSTM+attention model -- same inputs,
    same embeddings, same training loop, so the two can be compared head to head."""
    def __init__(self, embedding_matrix, hidden_dim=64, freeze_embeddings=False):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32), freeze=freeze_embeddings, padding_idx=0)
        embed_dim = embedding_matrix.shape[1]
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, nonlinearity="tanh")
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        rnn_out, _ = self.rnn(embedded)
        # masked mean pool over non-pad timesteps -- more stable than only using the final
        # hidden state, which for a vanilla RNN can be dominated by the last few tokens.
        mask = (x != 0).unsqueeze(-1).float()
        summed = torch.sum(rnn_out * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-8)
        pooled = summed / counts
        return self.fc(pooled).squeeze(-1)

print("Model class ready.")

Model class ready.


## Prepare data (same split as the LSTM notebook), run a small hyperparameter search, train final model

In [6]:
tokenized_all = [basic_tokenize(clean_text(t)) for t in df["text"]]
binary_labels = (df["label"] == FAKE_LABEL).astype(int).tolist()

# Identical split as 03_lstm_attention.ipynb: same test_size, same random_state=42,
# same stratification -- so train/val/test rows line up across both models.
train_tok, temp_tok, y_train_b, y_temp_b = train_test_split(
    tokenized_all, binary_labels, test_size=0.3, random_state=RANDOM_STATE, stratify=binary_labels)
val_tok, test_tok, y_val_b, y_test_b = train_test_split(
    temp_tok, y_temp_b, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp_b)

vocab = build_vocab(train_tok)
embedding_matrix = build_embedding_matrix(vocab, kv, EMBED_DIM)

X_train = [encode(t, vocab) for t in train_tok]
X_val = torch.tensor([encode(t, vocab) for t in val_tok], dtype=torch.long)
X_test = torch.tensor([encode(t, vocab) for t in test_tok], dtype=torch.long)
y_val_t = torch.tensor(y_val_b, dtype=torch.float32)
y_test_t = torch.tensor(y_test_b, dtype=torch.float32)

class_counts = [y_train_b.count(0), y_train_b.count(1)]
sample_weights = [1.0 / class_counts[l] for l in y_train_b]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(y_train_b), replacement=True)
train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.long), torch.tensor(y_train_b, dtype=torch.float32))
train_loader = DataLoader(train_ds, batch_size=16, sampler=sampler)

def train_and_eval(hidden_dim, lr, epochs):
    model = RNNClassifier(embedding_matrix, hidden_dim=hidden_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    cross_entropy_loss = nn.BCEWithLogitsLoss()  # binary cross-entropy = cross-entropy for 2 classes
    model.train()
    for epoch in range(epochs):
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = cross_entropy_loss(logits, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        val_preds = (torch.sigmoid(val_logits) > 0.5).float().numpy()
    return model, f1_score(y_val_t.numpy(), val_preds, average="macro", zero_division=0)

print("=== Hyperparameter search over variable parameters ===")
results = []
for cfg in [{"hidden_dim": 32, "lr": 1e-3}, {"hidden_dim": 64, "lr": 1e-3},
            {"hidden_dim": 64, "lr": 5e-4}, {"hidden_dim": 128, "lr": 5e-4}]:
    _, val_f1 = train_and_eval(cfg["hidden_dim"], cfg["lr"], epochs=5)
    results.append({**cfg, "val_macro_f1": val_f1})
    print(f"  hidden_dim={cfg['hidden_dim']:>4}  lr={cfg['lr']:<7}  val_macro_f1={val_f1:.3f}")

best = max(results, key=lambda r: r["val_macro_f1"])
print(f"\nBest config: {best}")

print(f"\n=== Final run: hidden_dim={best['hidden_dim']}, lr={best['lr']} ===")
final_model, _ = train_and_eval(best["hidden_dim"], best["lr"], epochs=15)
final_model.eval()
with torch.no_grad():
    test_logits = final_model(X_test)
    test_preds = (torch.sigmoid(test_logits) > 0.5).numpy()

print(f"Accuracy: {accuracy_score(y_test_t.numpy(), test_preds):.3f}\n")
print(classification_report(y_test_t.numpy(), test_preds, target_names=["AUTHENTIC", "FAKE"], zero_division=0))

Found pretrained vectors for 6582/8000 words (82.3%).
=== Hyperparameter search over variable parameters ===
  hidden_dim=  32  lr=0.001    val_macro_f1=0.677
  hidden_dim=  64  lr=0.001    val_macro_f1=0.678
  hidden_dim=  64  lr=0.0005   val_macro_f1=0.665
  hidden_dim= 128  lr=0.0005   val_macro_f1=0.658

Best config: {'hidden_dim': 64, 'lr': 0.001, 'val_macro_f1': 0.677629115319085}

=== Final run: hidden_dim=64, lr=0.001 ===
Accuracy: 0.836

              precision    recall  f1-score   support

   AUTHENTIC       0.91      0.90      0.90      1155
        FAKE       0.45      0.46      0.45       200

    accuracy                           0.84      1355
   macro avg       0.68      0.68      0.68      1355
weighted avg       0.84      0.84      0.84      1355



## Does this architecture match the U / W / V shape diagram?

The diagram is the classic Elman-RNN recurrence: `h_t = f(W e_t + U h_{t-1})`, `y_hat_t = softmax(V h_t)`, with `e_t` and `h_t` both `d x 1`, `W` and `U` both `d x d`, and `V` an `|V| x d` matrix that turns the hidden state into a full distribution over the vocabulary at every timestep -- i.e. it is the shape diagram for an RNN **language model** predicting the next token.

This notebook's `RNNClassifier` is a sequence **classifier** (FAKE vs AUTHENTIC), not a language model, so it is worth checking explicitly rather than assuming. The cell below pulls the real weight matrices out of the trained model and prints their shapes next to what the diagram says they should be.

In [7]:
print("=== This model's parameter shapes vs. the diagram's U / W / V shapes ===\n")

hidden_dim = best["hidden_dim"]
vocab_size = len(vocab)

W_ih = final_model.rnn.weight_ih_l0   # combines with e_t  -> diagram's "W"
W_hh = final_model.rnn.weight_hh_l0   # combines with h_{t-1} -> diagram's "U"
b_ih = final_model.rnn.bias_ih_l0
b_hh = final_model.rnn.bias_hh_l0
fc_w = final_model.fc.weight          # this model's output layer -> diagram's "V"
fc_b = final_model.fc.bias

rows = [
    ("e_t   (embedded token)",         f"{EMBED_DIM} x 1",        "d x 1"),
    ("h_t-1 / h_t (hidden state)",     f"{hidden_dim} x 1",       "d x 1"),
    ("W  = rnn.weight_ih_l0",          f"{tuple(W_ih.shape)}",    "d x d"),
    ("U  = rnn.weight_hh_l0",          f"{tuple(W_hh.shape)}",    "d x d"),
    ("V  = fc.weight (this model)",    f"{tuple(fc_w.shape)}",    "|V| x d"),
    ("y_hat (this model, per review)", f"{tuple(fc_b.shape)} -> 1 x 1", "|V| x 1"),
]
print(f"{'Component':32} {'This model':18} {'Diagram says'}")
for name, actual, expected in rows:
    flag = "OK " if str(actual).replace(" ", "") == expected.replace(" ", "").replace("d","").replace("|V|","") else "!! "
    print(f"{name:32} {actual:18} {expected}")

print(f"\n|V| (vocabulary size actually used for the embedding) = {vocab_size}")
print(f"d   (hidden_dim chosen by the search)                  = {hidden_dim}")
print(f"embedding dim (fastText)                                = {EMBED_DIM}")
print("Actual vocabulary size:", len(vocab))

=== This model's parameter shapes vs. the diagram's U / W / V shapes ===

Component                        This model         Diagram says
e_t   (embedded token)           300 x 1            d x 1
h_t-1 / h_t (hidden state)       64 x 1             d x 1
W  = rnn.weight_ih_l0            (64, 300)          d x d
U  = rnn.weight_hh_l0            (64, 64)           d x d
V  = fc.weight (this model)      (1, 64)            |V| x d
y_hat (this model, per review)   (1,) -> 1 x 1      |V| x 1

|V| (vocabulary size actually used for the embedding) = 8000
d   (hidden_dim chosen by the search)                  = 64
embedding dim (fastText)                                = 300
Actual vocabulary size: 8000


**Verdict: partially.**

- The recurrence itself matches: `h_t = tanh(W e_t + U h_{t-1} + b)` is exactly what `nn.RNN` computes, so the `U` and `W` roles in the diagram are real and present in this model.
- `W` is **not `d x d`** here: it's `hidden_dim x EMBED_DIM` (e.g. `64 x 300`), because the fastText embedding dimension and the chosen hidden size are different numbers. The diagram's `d x d` assumption only holds when the embedding dimension is set equal to `d`.
- `V` and `y_hat_t` **don't match at all**: the diagram's `V` is `|V| x d` and produces a softmax over the whole vocabulary at *every* timestep -- that's the shape diagram for a *language model* predicting the next word. This notebook's output layer is `1 x hidden_dim`, applied once to a pooled representation of the *whole* review, producing a single FAKE/AUTHENTIC logit -- a sequence classifier, not a per-token vocabulary predictor.
- `U` and `h_t` do match the `d x d` / `d x 1` shapes exactly, since both refer to the hidden state only.

So: same core recurrence cell as the diagram, different (and much smaller) output head, because the task here is binary classification rather than next-word prediction.

In [8]:
import os, joblib

os.makedirs("saved_rnn_model", exist_ok=True)
torch.save(final_model.state_dict(), "saved_rnn_model/rnn_model.pt")
joblib.dump(vocab, "saved_rnn_model/vocab.joblib")
joblib.dump({"hidden_dim": best["hidden_dim"], "embed_dim": EMBED_DIM, "max_seq_len": MAX_SEQ_LEN},
            "saved_rnn_model/config.joblib")
joblib.dump({"authentic": AUTHENTIC_LABEL, "fake": FAKE_LABEL}, "saved_rnn_model/label_map.joblib")
print("Saved to Drive -- saved_rnn_model/ -- reusable without retraining.")

Saved to Drive -- saved_rnn_model/ -- reusable without retraining.


## Live input/output testing

Loads the saved RNN (no retraining needed) and lets you type a review to get a live
FAKE / AUTHENTIC prediction with a confidence score -- same interface style as the demo notebook,
but backed by this RNN instead of the TF-IDF + Logistic Regression baseline.

In [15]:
import joblib

_vocab = joblib.load("saved_rnn_model/vocab.joblib")
_config = joblib.load("saved_rnn_model/config.joblib")
_label_map = joblib.load("saved_rnn_model/label_map.joblib")

_dummy_embedding_matrix = np.zeros((len(_vocab), _config["embed_dim"]), dtype=np.float32)
_inference_model = RNNClassifier(_dummy_embedding_matrix, hidden_dim=_config["hidden_dim"])
_inference_model.load_state_dict(torch.load("saved_rnn_model/rnn_model.pt", map_location="cpu"))
_inference_model.eval()

def predict(text):
    tokens = basic_tokenize(clean_text(text))
    ids = encode(tokens, _vocab, max_len=_config["max_seq_len"])
    x = torch.tensor([ids], dtype=torch.long)
    with torch.no_grad():
        logit = _inference_model(x)
        prob_fake = torch.sigmoid(logit).item()
    verdict = "FAKE" if prob_fake > 0.5 else "AUTHENTIC"
    confidence = prob_fake if verdict == "FAKE" else 1 - prob_fake
    return verdict, confidence

# Run this cell as many times as you like with different input
review = input("Enter a Bangla review: ")
verdict, confidence = predict(review)
print(f"Prediction: {verdict}  (confidence: {confidence:.1%})")

Enter a Bangla review: সেরা অফার পাস্তা খেলাম
Prediction: FAKE  (confidence: 100.0%)


## Comparison notes

Because this notebook mirrors the LSTM+attention notebook (same `cleaned_bfrd.csv`, same `random_state=42` split, same pretrained fastText embeddings, same loss and metrics), the `val_macro_f1` search table and the final accuracy / precision / recall / F1 / macro-F1 report above can be placed side by side with the LSTM notebook's output for the final model comparison in the report.